# 04. Testowanie kodu zależnego od czasu

## Cele

Po tym notebooku potrafisz:

* projektować interfejsy, które przyjmują czas w typach `<chrono>`, a nie w `int`-ach,
* wyjaśnić, dlaczego kod, który sam woła `steady_clock::now()`, trudno przetestować,
* przekazać zegar jako parametr szablonu i napisać własny zegar spełniający wymagania standardu,
* testować klasy zależne od czasu z `FakeClock` — bez usypiania testów i bez losowych porażek,
* opisać dwa praktyczne zastosowania: cache z czasem życia (TTL) i ogranicznik liczby zapytań (token bucket).

## Czas w interfejsach

Notebook 01 zaczynał od `blink_led(int)` kontra `blink_led(milliseconds)`. Ta sama zasada dotyczy każdego API:
parametr „timeout”, „okres”, „czas życia” powinien mieć typ odcinka czasu, a nie gołej liczby.
Biblioteka Abseil (Google) ujmuje to tak: *„Durations are unit-safe, making them the ideal type for interfaces
accepting timeouts or any other fixed-length span of time”*, a na liczby zamieniamy je tylko **na brzegu systemu** —
przy odczycie konfiguracji, zapisie do pliku, wywołaniu starego API
([Abseil: Time Programming](https://abseil.io/docs/cpp/guides/time#proper-time-hygiene)).

```cpp
#include <chrono>
#include <iostream>

class Connection {
public:
    explicit Connection(std::chrono::milliseconds timeout) : timeout_(timeout) {}
    std::chrono::milliseconds timeout() const { return timeout_; }
private:
    std::chrono::milliseconds timeout_;
};

int main() {
    using namespace std::chrono_literals;
    int timeout_ms_from_config = 2500;                                  // brzeg systemu: liczba z pliku
    Connection a{std::chrono::milliseconds{timeout_ms_from_config}};   // jednostka nadana raz, tutaj
    Connection b{3s};                                                   // w kodzie: literał z jednostką
    std::cout << a.timeout().count() << " ms, " << b.timeout().count() << " ms\n";
}
```

## Problem: kod, który sam patrzy na zegarek

Wyobraź sobie klasę, która w środku woła `std::chrono::steady_clock::now()`. Jak sprawdzić w teście,
że po 30 sekundach coś się zmienia?

* **Uśpić test na 30 s** — testy stają się wolne, a i tak nie wiadomo, ile *dokładnie* minęło.
* **Sprawdzać „mniej więcej”** (wynik między 29,9 s a 30,1 s) — test raz przechodzi, raz nie,
  zależnie od obciążenia maszyny. Sandor Dargo nazywa takie testy *„brittle, inaccurate”*
  ([Time in C++: Once More About Testing](https://www.sandordargo.com/blog/2026/01/28/clocks-part-9-once-more-about-testing#the-problem-of-time-and-testing)).

Książka *Software Engineering at Google* wymienia zegar systemowy jako typowe źródło niedeterminizmu w testach
i radzi zastąpić go zegarem, którym test steruje
([rozdz. 13, „Determinism”](https://abseil.io/resources/swe-book/html/ch13.html#determinism)).
Taki zamiennik to **fake** — lekka, działająca implementacja, która nie nadaje się do produkcji
([rozdz. 13, „Faking”](https://abseil.io/resources/swe-book/html/ch13.html#faking-id00042),
[Martin Fowler: TestDouble](https://martinfowler.com/bliki/TestDouble.html)).

## Zegar jako parametr szablonu

W `<chrono>` zegar to **typ**. Standard opisuje, co taki typ musi mieć
([\[time.clock.req\]](https://eel.is/c++draft/time.clock.req#1),
[cppreference: Clock](https://en.cppreference.com/cpp/named_req/Clock#Requirements)):

| Składowa | Znaczenie |
|---|---|
| `rep`, `period`, `duration` | typ licznika, okres tyknięcia, typ odcinka czasu |
| `time_point` | typ punktu w czasie tego zegara |
| `is_steady` | czy zegar nigdy się nie cofa |
| `static time_point now()` | bieżący odczyt |

Biblioteka standardowa sama przyjmuje zegar jako parametr szablonu — np. `condition_variable::wait_until`
działa z dowolnym zegarem ([cppreference](https://en.cppreference.com/cpp/thread/condition_variable/wait_until)).
Możemy zrobić tak samo: klasa dostaje `class Clock = std::chrono::steady_clock`. W programie wszystko zostaje
bez zmian, a test podstawia własny zegar. Przykład — prosty stoper:

```cpp
#include <chrono>
#include <iostream>
#include <thread>

template <class Clock = std::chrono::steady_clock>
class Stopwatch {
public:
    Stopwatch() : start_(Clock::now()) {}
    typename Clock::duration elapsed() const { return Clock::now() - start_; }
private:
    typename Clock::time_point start_;
};

int main() {
    using namespace std::chrono_literals;
    Stopwatch<> sw;  // domyślnie steady_clock
    std::this_thread::sleep_for(20ms);
    auto ms = std::chrono::duration_cast<std::chrono::milliseconds>(sw.elapsed());
    std::cout << "minęło co najmniej 20 ms: " << (ms >= 20ms) << '\n';
}
```

Uwaga na `typename`: w szablonie `Clock::time_point` i `Clock::duration` są *nazwami zależnymi* i przed typem
trzeba napisać `typename` ([cppreference: dependent names](https://en.cppreference.com/cpp/language/dependent_name#The_typename_disambiguator_for_dependent_names)).
C++20 pozwala go czasem pominąć, ale Clang obsługuje to dopiero od wersji 16
([C++20 compiler support](https://en.cppreference.com/cpp/compiler_support/20)) — piszemy więc `typename` zawsze
przed **typem**. I tylko przed typem: wywołanie funkcji statycznej to nie typ.

```cpp
#include <chrono>

template <class Clock>
typename Clock::time_point earliest() {
    return typename Clock::time_point::min();  // błąd: min() to funkcja, nie typ
}

int main() {
    (void)earliest<std::chrono::steady_clock>();
}
```

Poprawnie: `return Clock::time_point::min();`.

## `FakeClock`: zegar, którym steruje test

Sztuczny zegar to struktura z wymaganymi składowymi, w której „teraz” jest zwykłą zmienną. Test przesuwa czas
metodą `advance()`. Ponieważ `now()` musi być statyczne, stan zegara też jest statyczny — dlatego każdy test
powinien zaczynać od `reset()` (w GoogleTest: w `SetUp()` fikstury). Podobny „ręcznie przesuwany” zegar opisuje
Sandor Dargo ([Creating Your Own Clocks](https://www.sandordargo.com/blog/2026/01/14/clocks-part-7-custom-clocks#a-simple-example-a-manually-advanced-clock)).

```cpp
#include <chrono>
#include <gtest/gtest.h>

struct FakeClock {
    using duration = std::chrono::nanoseconds;
    using rep = duration::rep;
    using period = duration::period;
    using time_point = std::chrono::time_point<FakeClock>;
    static constexpr bool is_steady = true;

    static time_point now() { return current; }
    static void advance(duration d) { current += d; }
    static void reset() { current = time_point{}; }

    inline static time_point current{};
};

template <class Clock = std::chrono::steady_clock>
class Stopwatch {
public:
    Stopwatch() : start_(Clock::now()) {}
    typename Clock::duration elapsed() const { return Clock::now() - start_; }
private:
    typename Clock::time_point start_;
};

class StopwatchTest : public ::testing::Test {
protected:
    void SetUp() override { FakeClock::reset(); }
};

TEST_F(StopwatchTest, MeasuresExactlyWhatTheClockSays) {
    using namespace std::chrono_literals;
    Stopwatch<FakeClock> sw;
    FakeClock::advance(90min);  // półtorej godziny "mija" w mikrosekundę
    EXPECT_EQ(sw.elapsed(), 90min);
}

TEST_F(StopwatchTest, StartsAtZero) {
    Stopwatch<FakeClock> sw;
    EXPECT_EQ(sw.elapsed().count(), 0);
}
```

`EXPECT_EQ(sw.elapsed(), 90min)` porównuje odcinki czasu w różnych jednostkach — `<chrono>` przelicza je sam.
Przy porażce GoogleTest wypisuje odcinki czasu czytelnie dopiero w nowszych kompilatorach (z `operator<<` dla `<chrono>`);
na starszych zobaczysz zrzut bajtów — wtedy pomaga porównywanie `.count()` albo własny komunikat po `<<`.

**Alternatywa**: zamiast parametru szablonu klasa może przyjmować w konstruktorze funkcję zwracającą „teraz”,
np. `std::function<steady_clock::time_point()>`. Nie ma wtedy stanu statycznego, ale każde wywołanie kosztuje
wywołanie pośrednie, a typ zegara nie jest już częścią typu klasy. W tym laboratorium używamy parametru szablonu.

## Mierz od stałego punktu

Notebook 02 pokazał, że pętla z `sleep_until` liczy terminy od stałego punktu i nie gubi czasu.
Ta sama zasada dotyczy liczenia: jeśli wielokrotnie zaokrąglasz i sumujesz małe kawałki, reszty przepadają.
Stoper, który liczy od startu, ich nie gubi. Z `FakeClock` widać to dokładnie:

```cpp
#include <chrono>
#include <iostream>

struct FakeClock {
    using duration = std::chrono::nanoseconds;
    using rep = duration::rep;
    using period = duration::period;
    using time_point = std::chrono::time_point<FakeClock>;
    static constexpr bool is_steady = true;
    static time_point now() { return current; }
    static void advance(duration d) { current += d; }
    inline static time_point current{};
};

int main() {
    using namespace std::chrono;
    const auto start = FakeClock::now();
    auto last = start;
    milliseconds summed{0};

    for (int lap = 0; lap < 4; ++lap) {
        FakeClock::advance(1500us);                               // każde okrążenie: 1,5 ms
        summed += duration_cast<milliseconds>(FakeClock::now() - last);  // 1 ms, reszta przepada
        last = FakeClock::now();
    }
    auto from_start = duration_cast<milliseconds>(FakeClock::now() - start);
    std::cout << "suma zaokrąglonych okrążeń: " << summed.count() << " ms\n";
    std::cout << "od stałego punktu startu:   " << from_start.count() << " ms\n";
}
```

## Zastosowanie 1: cache z czasem życia (TTL)

Klient API nie musi pytać serwera o to samo co sekundę. Odpowiedź można zapamiętać razem z chwilą,
w której przestaje być aktualna: `wygasa = teraz + ttl`
([Wikipedia: Time to live](https://en.wikipedia.org/wiki/Time_to_live)). Do tego potrzebny jest `steady_clock` —
zmiana zegara systemowego nie może nagle „odświeżyć” albo „przeterminować” wpisu.

Ważna jest granica: czy wpis z TTL 30 s jest jeszcze ważny po **dokładnie** 30 sekundach? Cache HTTP
odpowiada „nie”: odpowiedź jest świeża, dopóki jej wiek jest **mniejszy** niż czas życia
([RFC 9111, §4.2](https://www.rfc-editor.org/rfc/rfc9111.html#section-4.2)). Takie granice to idealny materiał na
testy z `FakeClock` — przesunięcie zegara o dokładnie 30 s jest w nich trywialne.

## Zastosowanie 2: ogranicznik liczby zapytań (token bucket)

Serwisy API ograniczają, ile zapytań wolno wysłać w danym czasie — tak robią np. Stripe i Amazon API Gateway
([Stripe: Scaling your API with rate limiters](https://stripe.com/blog/rate-limiters),
[AWS: request throttling](https://docs.aws.amazon.com/apigateway/latest/developerguide/api-gateway-request-throttling.html)).
Klient, który sam pilnuje limitu, nie dostaje odmów od serwera. Popularny algorytm to **wiadro z żetonami**
([Wikipedia: Token bucket](https://en.wikipedia.org/wiki/Token_bucket#Algorithm)):

* wiadro mieści najwyżej `b` żetonów i na początku jest pełne,
* co stały okres (np. co 1 s) przybywa jeden żeton; gdy wiadro jest pełne, nadmiarowy żeton przepada,
* każde zapytanie zabiera jeden żeton; gdy żetonów brak, zapytanie trzeba odłożyć.

Przykład dla `b = 2` i jednego żetonu co 5 sekund:

| Czas | Zdarzenie | Żetony po |
|---|---|---|
| 0 s | start, wiadro pełne | 2 |
| 0 s | 2 zapytania | 0 |
| 0 s | 3. zapytanie — odmowa | 0 |
| 5 s | przybył żeton | 1 |
| 5 s | zapytanie | 0 |
| dzień później | przez dobę przybywały żetony, ale wiadro mieści 2 | 2 |

Dzięki temu dozwolona jest krótka „seria” zapytań, ale średnie tempo nie przekracza jednego na 5 sekund.

## Ćwiczenie 4

*Treść ćwiczenia — w przygotowaniu.*

## Ćwiczenie 5

*Treść ćwiczenia — w przygotowaniu.*

## Linki i materiały

* [C++ Core Guidelines, I.4](https://isocpp.github.io/CppCoreGuidelines/CppCoreGuidelines#ri-typed),
  [Abseil: Time Programming](https://abseil.io/docs/cpp/guides/time#time-durations)
* [cppreference: wymagania Clock](https://en.cppreference.com/cpp/named_req/Clock#Requirements),
  [TrivialClock](https://en.cppreference.com/cpp/named_req/TrivialClock#Requirements);
  standard: [\[time.clock.req\]](https://eel.is/c++draft/time.clock.req#4)
* Sandor Dargo, seria „Time in C++”:
  [własne zegary](https://www.sandordargo.com/blog/2026/01/14/clocks-part-7-custom-clocks#why-would-you-want-a-custom-clock),
  [testowanie](https://www.sandordargo.com/blog/2026/01/28/clocks-part-9-once-more-about-testing#the-problem-of-time-and-testing)
* *Software Engineering at Google*: [rozdz. 12 „Unit Testing”](https://abseil.io/resources/swe-book/html/ch12.html),
  [rozdz. 13 „Test Doubles”](https://abseil.io/resources/swe-book/html/ch13.html#determinism)
* [Wikipedia: Token bucket](https://en.wikipedia.org/wiki/Token_bucket#Overview),
  [RFC 9111 HTTP Caching, §4.2](https://www.rfc-editor.org/rfc/rfc9111.html#section-4.2)

---

[← 03. Kalendarz](../03_calendar/03_calendar.ipynb) | [Strona główna](../_start.ipynb) | [05. Dla ciekawych →](../05_extras/05_extras.ipynb)